In [ ]:
import re
import html
from pathlib import Path

INPUT_DIR = "transcripts"
OUTPUT_FILE = "corpus.txt"


def clean_vtt(text):
    text = html.unescape(text)

    lines = text.splitlines()
    cleaned = []
    seen_lines = set()

    for line in lines:
        line = line.strip()

        if not line:
            continue

        # Skip metadata
        if line.startswith(("WEBVTT", "Kind:", "Language:")):
            continue

        # Skip timestamp lines
        if "-->" in line:
            continue

        # Skip pure sound-effect lines
        if re.fullmatch(r"(?:\[[^\]]*\]\s*)+", line):
            continue

        # Remove embedded sound-effect tags
        line = re.sub(r"\[[^\]]*\]", "", line)

        # Remove inline timestamps
        line = re.sub(r"<\d{2}:\d{2}:\d{2}\.\d+>", "", line)

        # Remove caption tags
        line = re.sub(r"</?c>", "", line)

        # Remove speaker markers
        line = re.sub(r"^>>\s*", "", line)

        # Remove remaining HTML/XML tags
        line = re.sub(r"<[^>]+>", "", line)

        # Normalize whitespace
        line = re.sub(r"\s+", " ", line).strip()

        if len(line) < 2:
            continue

        # Deduplicate repeated lines within file
        if line not in seen_lines:
            cleaned.append(line)
            seen_lines.add(line)

    return cleaned


all_text = []
seen_filenames = set()

processed = 0
duplicates = 0

for file in sorted(Path(INPUT_DIR).rglob("*")):

    if not file.is_file():
        continue

    filename = file.name

    if filename in seen_filenames:
        duplicates += 1
        print(f"[DUPLICATE SKIPPED] {file}")
        continue

    seen_filenames.add(filename)

    try:
        text = file.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        cleaned = clean_vtt(text)

        if cleaned:
            all_text.append(" ".join(cleaned))

        processed += 1

    except Exception as e:
        print(f"[ERROR] {file}: {e}")

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write("\n\n".join(all_text))

print("\n===== SUMMARY =====")
print(f"Processed files      : {processed:,}")
print(f"Duplicates skipped   : {duplicates:,}")
print(f"Output corpus        : {OUTPUT_FILE}")

usage: ipykernel_launcher.py [-h] [--input INPUT] [--output OUTPUT]
                             [--threshold THRESHOLD] [--report REPORT]
ipykernel_launcher.py: error: unrecognized arguments: --f=/run/user/29999/jupyter/runtime/kernel-v3482c232dd308cde966aafd3bffa6a058f40c0685.json


SystemExit: 2

In [3]:
!pip install datasketch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.9/99.9 kB 4.9 MB/s eta 0:00:00


command: yt-dlp --skip-download --write-auto-subs --sub-lang ar --sub-format vtt   "https://www.youtube.com/@HekayatWaZekrayat"